In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(stringr)
  library(Matrix)
  library(edgeR)
  library(ComplexHeatmap)
  library(circlize)
  library(RColorBrewer)
  library(grid)
})

options(future.globals.maxSize = 120 * 1024^3)
set.seed(42)


## 1. Paths and analysis parameters


In [ ]:
project_dir <- "/oak/stanford/projects/kibr/Reorganizing/Projects/Andi/vascular_atlas"
setwd(project_dir)

h5ad_file <- "./Results/Revision_3/Capi_temp_processed.h5ad"

# Save the supplied annotation CSV here, or change this path.
gene_file <- "./Results/Revision_3/transport_marker_function_annotated_27.csv"

out_pdf <- "./Results/Revision_3/Figures/Capillary_transport_heatmap_per_region.pdf"
out_logcpm_csv <- "./Results/Revision_3/Capillary_transport_logCPM_by_region.csv"
out_zscore_csv <- "./Results/Revision_3/Capillary_transport_zscore_by_region.csv"

# Your original comment said "< 3 cells", whereas the old code used < 100
# at the whole-region level. This notebook filters DONOR x REGION pseudobulk
# samples. Change 3L to 100L if 100 nuclei per donor-region sample is intended.
min_cells_per_donor_region <- 3L

z_limit <- 2.5


## 2. Load the capillary Seurat object


In [ ]:
obj_oi <- schard::h5ad2seurat(h5ad_file)

DefaultAssay(obj_oi) <- "RNA"

required_meta <- c("region_name", "individualID", "region_layer",'region_abb')
missing_meta <- setdiff(required_meta, colnames(obj_oi[[]]))
if (length(missing_meta) > 0) {
  stop("Missing metadata columns: ", paste(missing_meta, collapse = ", "))
}

print(obj_oi)
cat("Donors:", length(unique(obj_oi$individualID)), "\n")
cat("Regions before filtering:", length(unique(obj_oi$region_name)), "\n")



## 3. Filter low-cell donor-region samples


In [ ]:
meta <- obj_oi[[]]
meta$cell_id <- rownames(meta)

sample_qc <- meta %>%
  count(region_abb, individualID, name = "n_cells") %>%
  arrange(region_abb, individualID)

print(sample_qc)

valid_samples <- sample_qc %>%
  filter(n_cells >= min_cells_per_donor_region)

keep_cells <- meta %>%
  inner_join(
    valid_samples %>% select(region_abb, individualID),
    by = c("region_abb", "individualID")
  ) %>%
  pull(cell_id)

obj_oi <- subset(obj_oi, cells = keep_cells)

cat("Cells retained:", ncol(obj_oi), "\n")
cat("Donors retained:", length(unique(obj_oi$individualID)), "\n")
cat("Regions retained:", length(unique(obj_oi$region_abb)), "\n")

# Number of retained donors contributing to each region
donors_per_region <- obj_oi[[]] %>%
  distinct(region_abb, individualID) %>%
  count(region_abb, name = "n_donors")

print(donors_per_region)


## 4. Load the 27-gene functional annotation


In [ ]:
gene_df <- read.csv(
  gene_file,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

required_gene_cols <- c(
  "gene",
  "function_group",
  "function_group_order",
  "concise_function"
)

missing_gene_cols <- setdiff(required_gene_cols, colnames(gene_df))
if (length(missing_gene_cols) > 0) {
  stop("Gene annotation file is missing: ",
       paste(missing_gene_cols, collapse = ", "))
}

gene_df <- gene_df %>%
  arrange(function_group_order, gene_order_within_group)

stopifnot(!anyDuplicated(gene_df$gene))
cat("Genes in annotation file:", nrow(gene_df), "\n")

## 5. Donor-level pseudobulk aggregation

Using a synthetic `PB###` identifier avoids parsing region names from Seurat-generated
column names. This is safer when region names themselves contain underscores.


In [ ]:
sample_meta <- obj_oi[[]] %>%
  distinct(region_abb, individualID) %>%
  arrange(region_abb, individualID) %>%
  mutate(pb_id = sprintf("PB%03d", row_number()))

cell_meta <- obj_oi[[]]

cell_key <- paste(cell_meta$region_abb, cell_meta$individualID, sep = "|||")
sample_key <- paste(sample_meta$region_abb, sample_meta$individualID, sep = "|||")

obj_oi$pb_id <- sample_meta$pb_id[match(cell_key, sample_key)]

if (anyNA(obj_oi$pb_id)) {
  stop("Failed to assign pseudobulk IDs to some cells.")
}

agg <- AggregateExpression(
  obj_oi,
  group.by = "pb_id",
  assays = "RNA",
  slot = "counts",
  return.seurat = FALSE
)

mtx <- as.matrix(agg$RNA)

sample_meta <- sample_meta[match(colnames(mtx), sample_meta$pb_id), , drop = FALSE]

if (!identical(sample_meta$pb_id, colnames(mtx))) {
  stop("Pseudobulk sample metadata do not match the count-matrix columns.")
}

dim(mtx)


## 6. TMM-normalized logCPM

TMM normalization is preferable to manually dividing by raw library size because it
reduces composition bias between donor-region pseudobulk samples.


In [ ]:
dge <- DGEList(counts = mtx)
dge <- calcNormFactors(dge, method = "TMM")

logCPM_all <- cpm(
  dge,
  log = TRUE,
  prior.count = 1
)

missing_genes <- setdiff(gene_df$gene, rownames(logCPM_all))
if (length(missing_genes) > 0) {
  warning(
    "Genes absent from the RNA assay and omitted: ",
    paste(missing_genes, collapse = ", ")
  )
}

gene_plot_df <- gene_df %>%
  filter(gene %in% rownames(logCPM_all)) %>%
  arrange(function_group_order, gene_order_within_group)

logCPM <- logCPM_all[gene_plot_df$gene, , drop = FALSE]

## 7. Average the four donors to one column per brain region

The average is calculated from donor-level logCPM, so each retained donor contributes
equally to a region rather than regions with more nuclei receiving greater weight.


In [ ]:
region_levels <- sort(unique(sample_meta$region_abb))

logCPM_region <- vapply(
  region_levels,
  FUN = function(region_i) {
    idx <- which(sample_meta$region_abb == region_i)
    rowMeans(logCPM[, idx, drop = FALSE], na.rm = TRUE)
  },
  FUN.VALUE = numeric(nrow(logCPM))
)

rownames(logCPM_region) <- rownames(logCPM)
colnames(logCPM_region) <- region_levels

# Gene-wise Z-score across regions
logCPM_region_z <- t(scale(t(logCPM_region)))
logCPM_region_z[!is.finite(logCPM_region_z)] <- 0

stopifnot(identical(rownames(logCPM_region_z), gene_plot_df$gene))

dim(logCPM_region_z)

## 8. Export region-level expression matrices


In [ ]:
write.csv(
  data.frame(gene = rownames(logCPM_region), logCPM_region, check.names = FALSE),
  out_logcpm_csv,
  row.names = FALSE
)

write.csv(
  data.frame(gene = rownames(logCPM_region_z), logCPM_region_z, check.names = FALSE),
  out_zscore_csv,
  row.names = FALSE
)



## 9. Region annotations


In [ ]:
region_info <- obj_oi[[]] %>%
  distinct(region_abb, region_layer)

duplicated_region_layer <- region_info %>%
  count(region_abb) %>%
  filter(n > 1)

if (nrow(duplicated_region_layer) > 0) {
  stop("Some regions map to more than one region_layer: ",
       paste(duplicated_region_layer$region_abb, collapse = ", "))
}

region_info <- region_info[match(colnames(logCPM_region_z), region_info$region_abb), ]

if (anyNA(region_info$region_abb)) {
  stop("Missing region-layer annotation for one or more heatmap columns.")
}

region_layer_colors <- c(
  "Cortex" = "#009E73",
  "Brainstem" = "#D55E00",
  "Cerebellum" = "#046299",
  "Limbic" = "#03B8D8",
  "Watershed" = "#E69F00",
  "White Matter Tracts" = "#CC79A7"
)

missing_layers <- setdiff(
  unique(region_info$region_layer),
  names(region_layer_colors)
)

if (length(missing_layers) > 0) {
  region_layer_colors <- c(
    region_layer_colors,
    setNames(rep("#999999", length(missing_layers)), missing_layers)
  )
}

ha_top <- HeatmapAnnotation(
  Region_layer = region_info$region_layer,
  col = list(Region_layer = region_layer_colors),
  annotation_name_side = "left",
  show_legend = TRUE
)


## 10. Functional row groups

Rows are **split by biological function and clustered only within each functional
slice**. This prevents, for example, an ABC efflux pump from being intermingled
with a receptor simply because their regional expression patterns correlate.


In [ ]:
function_levels <- gene_plot_df %>%
  distinct(function_group_order, function_group) %>%
  arrange(function_group_order) %>%
  pull(function_group)

row_function <- factor(
  gene_plot_df$function_group,
  levels = function_levels
)

function_colors <- setNames(
  brewer.pal(length(function_levels), "Set2"),
  function_levels
)

ha_left <- rowAnnotation(
  Function = row_function,
  col = list(Function = function_colors),
  show_annotation_name = FALSE
)


## 11. Draw the heatmap


In [ ]:
heat_col <- colorRamp2(
  c(-z_limit, 0, z_limit),
  c("#5387BE", "#F7F7F7", "#B31F2C")
)

ht <- Heatmap(
  logCPM_region_z,
  name = "Gene-wise\nZ-score",
  col = heat_col,

  # Genes are grouped by function, then clustered within function.
  row_split = row_function,
  cluster_rows = TRUE,
  cluster_row_slices = FALSE,
  row_title_rot = 0,
  row_gap = unit(2, "mm"),

  # Regions are clustered from their 27-gene profiles.
  cluster_columns = TRUE,

  left_annotation = ha_left,
  top_annotation = ha_top,

  show_row_names = TRUE,
  row_names_gp = gpar(fontsize = 9),

  show_column_names = TRUE,
  column_names_rot = 90,
  column_names_gp = gpar(fontsize = 8),

  show_heatmap_legend = TRUE,
  heatmap_legend_param = list(
    at = c(-2, -1, 0, 1, 2),
    title = "Gene-wise\nZ-score"
  ),

  use_raster = TRUE,
  border = FALSE
)

options(repr.plot.width = 15,repr.plot.height = 6)
draw(
  ht,
  heatmap_legend_side = "right",
  annotation_legend_side = "right"
)


## 12. Save publication PDF


In [ ]:
dir.create(dirname(out_pdf), recursive = TRUE, showWarnings = FALSE)

pdf(out_pdf, width = 15, height = 6, useDingbats = FALSE)
draw(
  ht,
  heatmap_legend_side = "right",
  annotation_legend_side = "right"
)
dev.off()

cat("Saved:", out_pdf, "\n")
cat("Saved:", out_logcpm_csv, "\n")
cat("Saved:", out_zscore_csv, "\n")
